# Weeks 3+ — Working with the full release (~79M rows) without downloading 79M rows

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sarahnjunge/starter-notebooks/blob/main/notebooks/03_working_with_the_full_release.ipynb?flush_cache=true)

Notebooks 01–02 used the small starter CSV that ships with this repo. Your lane and capstone work
run on the **full pseudonymized warehouse release**: ~17 months of daily search performance for
~70 clients, plus a query-level table. It is hosted as Parquet on Hugging Face, and the trick of
this notebook is that you **never download or load the whole thing** — DuckDB reads only the
columns and partitions your SQL touches.

By the end you will have:
1. Connected DuckDB to the hosted release and listed every table.
2. Pulled a **feature table you designed** (aggregates per content item) into pandas.
3. Trained a quick scikit-learn model on features you built from 79M rows — on a free Colab CPU.

**Before you start (one-time, ~2 minutes):**
1. Create a free [Hugging Face account](https://huggingface.co/join).
2. Open the dataset page ([`FlyRank/internship-warehouse`](https://huggingface.co/datasets/FlyRank/internship-warehouse)) and **request access** (instant after you accept the data-use terms). **Accept the terms in your browser first — the token below 401s until access is granted (usually instant).**
3. Create a **read** token at [Settings → Access Tokens](https://huggingface.co/settings/tokens). **Never paste the token into a code cell** — your repo is public; use the `getpass` prompt below (or Colab's 🔑 Secrets panel).


### 1) Data contract — Content Decline Prediction

**1. What does one row mean?**  
One row represents one content item for one client at a particular reporting date in the search-performance panel.

**2. Which table(s) will I use?**  
I will use `fact_daily` for content-level daily search performance and `fact_query_90d` for historical query-level signals.

**3. What time window will I use?**  
I will develop and verify the contract using March 2026 as a mid-panel development month, using only information available before the prediction window.

**4. What will I predict?**  
I will predict whether a content item's search impressions decline by more than 20% in the future outcome window compared with its preceding historical window.

**5. What will I deliberately exclude?**  
I will exclude any feature calculated from the future outcome window, including future impressions, future clicks, or future ranking position, because these would not be knowable at the decision moment.

In [1]:
%pip -q install duckdb huggingface_hub


In [1]:
import os, getpass

# CI and power users set HF_TOKEN in the environment; everyone else gets the safe prompt.
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')


Paste your Hugging Face READ token (hf_...): ··········


## 1. Connect DuckDB to the release

DuckDB speaks `hf://` natively. The secret below authenticates every query; after that the
release behaves like a set of local tables.


In [2]:
import duckdb

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':                f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':                f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':                 f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
    'fact_daily_sample':          f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",
    'fact_query_90d':             f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:22} {n:>12,} rows')


dim_clients                     104 rows
dim_content                 519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

fact_daily               78,835,655 rows
fact_daily_sample        11,694,072 rows
fact_query_90d            2,414,248 rows


That count over the daily fact touched **Parquet metadata, not data** — it finished in seconds
even though the table has ~79M rows. That is the whole workflow: push the heavy lifting into
DuckDB SQL, bring only small results into pandas.

## 2. Know your panel before you model it

History depth **differs per client** (an *unbalanced panel*). `dim_clients` tells you exactly
what each client has — check it before designing any time window.


In [3]:
clients = con.sql(f"""
    SELECT client_hash_id, access_profile, gsc_data_start, ga4_data_start
    FROM {TABLES['dim_clients']}
    ORDER BY gsc_data_start NULLS LAST
""").df()

print('clients with 12+ months of GSC history:',
      (clients['gsc_data_start'] <= clients['gsc_data_start'].dropna().max() - __import__('pandas').Timedelta(days=365)).sum())
clients.head(10)


clients with 12+ months of GSC history: 4


,client_hash_id,access_profile,gsc_data_start,ga4_data_start
0,client_9958f0a7ae1df715,gsc_and_ga4,2025-01-27,2025-10-29
1,client_ff644d8251367cbb,gsc_and_ga4,2025-01-27,2025-10-29
2,client_73cda7b4e4f265ea,gsc_and_ga4,2025-02-11,2026-03-24
3,client_fef1a8f436438636,gsc_and_ga4,2025-03-11,2026-03-06
4,client_62f4a7e64f5e0096,gsc_only,2025-06-07,NaT
5,client_b10cb2997d0c7c86,gsc_and_ga4,2025-06-18,2025-11-15
6,client_65de48885f4ef01b,gsc_and_ga4,2025-06-21,2026-02-19
7,client_c182d11e4862a37d,gsc_and_ga4,2025-06-21,2026-02-20
8,client_3197e6291363b4db,gsc_and_ga4,2025-06-29,2025-11-09
9,client_625b6439094e23e4,gsc_and_ga4,2025-07-01,2026-02-19


## 3. Build features with SQL, not with RAM

The pattern for every lane: **aggregate per content item inside DuckDB**, then hand the small
result to pandas/sklearn. Here: momentum features from the last 60 days of the panel.

**This is the heaviest cell in the notebook — expect 2–6 minutes on Colab.** It downloads ~2 months of column data over the network (RAM stays tiny; that's the point). If it runs past ~10 minutes or errors with `HTTP 429`, re-run this section against `TABLES['fact_daily_sample']` and save the full table for your final pass.


In [4]:
features = con.sql(f"""
    WITH bounds AS (
        SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}
    ),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last45,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 45 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev45,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last45,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END)       AS pos_last45,
               STDDEV(CASE WHEN f.report_date > b.end_d - INTERVAL 45 DAY THEN f.gsc_avg_position END)     AS position_volatility
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 90 DAY
        GROUP BY 1, 2
        HAVING imp_prev45 >= 150
    )
    SELECT * FROM windowed
""").df()

print(f'{len(features):,} content items with enough history')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

111,067 content items with enough history


,client_hash_id,content_hash_id,imp_last45,imp_prev45,clk_last45,pos_last45,position_volatility
0,client_e547b89c05043229,content_25dfa3e39bc37247,600.0,676.0,3.0,11.338476,12.620193
1,client_e547b89c05043229,content_bab118937886d46a,216.0,253.0,0.0,22.903712,13.773609
2,client_e547b89c05043229,content_0587243c78e9468a,77.0,151.0,0.0,31.915365,21.106994
3,client_e547b89c05043229,content_5d6131702b65bee6,124.0,233.0,1.0,14.435897,14.635399
4,client_e547b89c05043229,content_a64be0ba11772089,495.0,610.0,2.0,18.775065,11.553878


## 4. Add query-level signals

`fact_content_query_90d` describes **how a page earns its impressions**: across how many
distinct queries, how concentrated, how much sits in the rare/anonymized tail. One page ranking
for 40 queries is a different animal from one page ranking for 2.


In [5]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

qsignals['top_query_share'] = qsignals['top_query_impressions'] / qsignals['kept_impressions']
data = features.merge(qsignals, on='content_hash_id', how='left')
print(f'joined: {len(data):,} rows')
data.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

joined: 111,067 rows


,client_hash_id,content_hash_id,imp_last45,imp_prev45,clk_last45,pos_last45,position_volatility,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,top_query_share
0,client_e547b89c05043229,content_25dfa3e39bc37247,600.0,676.0,3.0,11.338476,12.620193,5.0,0.038401,0.897335,34.0,82.0,0.414634
1,client_e547b89c05043229,content_bab118937886d46a,216.0,253.0,0.0,22.903712,13.773609,1.0,0.168443,0.528785,142.0,142.0,1.000000
2,client_e547b89c05043229,content_0587243c78e9468a,77.0,151.0,0.0,31.915365,21.106994,1.0,0.118421,0.763158,27.0,27.0,1.000000
3,client_e547b89c05043229,content_5d6131702b65bee6,124.0,233.0,1.0,14.435897,14.635399,2.0,0.162465,0.767507,14.0,25.0,0.560000
4,client_e547b89c05043229,content_a64be0ba11772089,495.0,610.0,2.0,18.775065,11.553878,1.0,0.081448,0.902262,18.0,18.0,1.000000


## 5. A first honest model

Same shape as notebook 02: define a label, hold out data, compare against a dumb baseline.
Label: *did impressions decline by more than 20% month-over-month?* — built only from columns
that exist **before** the window we predict. (Momentum features from the last 30 days predicting
a label defined on those same 30 days would be leakage — so here the features come from the
prev-30 window and query-mix, and the label from the last-30 outcome.)


In [6]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last45'] < 0.8 * data['imp_prev45']).astype(int)

feature_cols = ['imp_prev45', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'position_volatility' ]
model_data = data.dropna(subset=feature_cols)
X, y = model_data[feature_cols], model_data['is_declining']
groups = model_data['client_hash_id']

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)
train_idx, test_idx = next(gss.split(X, y, groups))

X_tr = X.iloc[train_idx]
X_te = X.iloc[test_idx]

y_tr = y.iloc[train_idx]
y_te = y.iloc[test_idx]
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)

print(f'base rate (always predict majority): {max(y_te.mean(), 1 - y_te.mean()):.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))


base rate (always predict majority): 0.532
              precision    recall  f1-score   support

           0      0.866     0.588     0.701      9379
           1      0.717     0.920     0.806     10650

    accuracy                          0.765     20029
   macro avg      0.792     0.754     0.753     20029
weighted avg      0.787     0.765     0.757     20029



In [8]:
print(data.columns.tolist())

['client_hash_id', 'content_hash_id', 'imp_last45', 'imp_prev45', 'clk_last45', 'pos_last45', 'position_volatility', 'visible_queries', 'rare_share', 'anon_share', 'top_query_impressions', 'kept_impressions', 'top_query_share', 'is_declining']


In [7]:
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
            AS distinct_client_content_date,
        COUNT(*) -
        COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR))
            AS duplicate_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_client_content_date,duplicate_rows
0,9841378,9841378,0


In [8]:
date_span_check = con.sql(f"""
    SELECT
        COUNT(*) AS row_count,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

date_span_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [10]:
con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {TABLES['fact_daily']}
    LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [11]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,available_rows
0,9841378,3611061


In [12]:
con.sql(f"""
    DESCRIBE
    SELECT *
    FROM {TABLES['fact_query_90d']}
    LIMIT 1
""").df()

,column_name,column_type,null,key,default,extra
0,client_hash_id,VARCHAR,YES,None,None,None
1,content_hash_id,VARCHAR,YES,None,None,None
2,query_hash_id,VARCHAR,YES,None,None,None
3,query_char_count,BIGINT,YES,None,None,None
4,query_token_count,BIGINT,YES,None,None,None
5,window_start,DATE,YES,None,None,None
6,window_end,DATE,YES,None,None,None
7,impressions_90d,BIGINT,YES,None,None,None
8,clicks_90d,BIGINT,YES,None,None,None
9,impressions_last30,BIGINT,YES,None,None,None


In [17]:
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS imp_prev45,

        SUM(gsc_clicks) AS clk_prev45,

        SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)
            AS ctr_prev45,

        STDDEV(gsc_avg_position) AS position_volatility,

        AVG(
            CASE
                WHEN DAYOFWEEK(report_date) IN (0, 6)
                THEN 1.0
                ELSE 0.0
            END
        ) AS weekend_share

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-01-15'
      AND report_date < DATE '2026-03-01'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id

    HAVING SUM(gsc_impressions) >= 150
""").df()

print(f"Feature rows: {len(features):,}")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature rows: 79,428


,client_hash_id,content_hash_id,imp_prev45,clk_prev45,ctr_prev45,position_volatility,weekend_share
0,client_3ffa76342f366962,content_0e92904e80e0ebf2,583.0,27.0,0.046312,2.388860,0.272727
1,client_3ffa76342f366962,content_8ffa6201737fd638,370.0,14.0,0.037838,3.542572,0.295455
2,client_3ffa76342f366962,content_3d4c0a897ebc9abd,386.0,85.0,0.220207,3.415949,0.272727
3,client_3ffa76342f366962,content_efcc94a28625575d,244.0,0.0,0.000000,2.504506,0.272727
4,client_3ffa76342f366962,content_99407cb063b988d2,310.0,3.0,0.009677,4.021600,0.255814


### Five features and availability

| Feature | Available when? |
|---|---|
| `imp_prev45` | Knowable at the decision moment because it is calculated only from GSC impressions in the preceding 45-day historical window. |
| `clk_prev45` | Knowable at the decision moment because it is calculated only from historical GSC clicks in the preceding 45-day window. |
| `ctr_prev45` | Knowable at the decision moment because it is calculated from historical clicks and impressions before the decision moment. |
| `position_volatility` | Knowable at the decision moment because it measures variation in historical average search position before the decision moment. |
| `weekend_share` | Knowable at the decision moment because it is calculated from the historical distribution of observations across weekdays and weekends. |

In [18]:
query_dates = con.sql(f"""
    SELECT
        MIN(window_start) AS earliest_window_start,
        MAX(window_end) AS latest_window_end,
        COUNT(*) AS total_rows
    FROM {TABLES['fact_query_90d']}
""").df()

query_dates

,earliest_window_start,latest_window_end,total_rows
0,2026-04-02,2026-06-30,2414248


In [19]:
features.info()
features.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 79428 entries, 0 to 79427
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   client_hash_id       79428 non-null  object 
 1   content_hash_id      79428 non-null  object 
 2   imp_prev45           79428 non-null  float64
 3   clk_prev45           79428 non-null  float64
 4   ctr_prev45           79428 non-null  float64
 5   position_volatility  79427 non-null  float64
 6   weekend_share        79428 non-null  float64
dtypes: float64(5), object(2)
memory usage: 4.2+ MB


,0
client_hash_id,0
content_hash_id,0
imp_prev45,0
clk_prev45,0
ctr_prev45,0
position_volatility,1
weekend_share,0


In [22]:
future_outcome = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS imp_future30

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'
      AND gsc_data_available IS TRUE

    GROUP BY
        client_hash_id,
        content_hash_id
""").df()

print(f"Future outcome rows: {len(future_outcome):,}")
future_outcome.head()

data = features.merge(
    future_outcome,
    on=['client_hash_id', 'content_hash_id'],
    how='inner'
)

data['is_declining'] = (
    (data['imp_future30'] / 30)
    < 0.8 * (data['imp_prev45'] / 45)
).astype(int)

print("Model rows:", len(data))
print("Declining rate:", data['is_declining'].mean())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future outcome rows: 176,738
Model rows: 75460
Declining rate: 0.18869599787967134


In [23]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score

feature_cols = [
    'imp_prev45',
    'clk_prev45',
    'ctr_prev45',
    'position_volatility',
    'weekend_share'
]

model_data = data.dropna(
    subset=feature_cols + ['is_declining']
).copy()

X = model_data[feature_cols]
y = model_data['is_declining']
groups = model_data['client_hash_id']

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

honest_score = balanced_accuracy_score(y_test, pred)

print(f"Honest balanced accuracy: {honest_score:.4f}")

Honest balanced accuracy: 0.5132


In [24]:
leaky_cols = feature_cols + ['imp_future30']

X_leaky = model_data[leaky_cols]

X_train_l = X_leaky.iloc[train_idx]
X_test_l = X_leaky.iloc[test_idx]

leaky_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

leaky_model.fit(X_train_l, y_train)

leaky_pred = leaky_model.predict(X_test_l)

leaky_score = balanced_accuracy_score(y_test, leaky_pred)

print(f"Leaky balanced accuracy: {leaky_score:.4f}")

Leaky balanced accuracy: 0.9778


## 4) Deliberate leakage check

I deliberately added `imp_future30` as a feature even though it is derived
from the future outcome window.

| Model | Balanced accuracy |
|---|---:|
| Honest model | 0.5132 |
| Leaky model | 0.9778 |

The leaky model's score increased dramatically from 0.5132 to 0.9778.
This confirms that `imp_future30` contains information about the target
that would not be available at the decision moment.

`imp_future30` was therefore removed from the feature set. The retained
honest balanced accuracy is **0.5132**.

### Named limitation

The current historical features have limited predictive power for the
decline label, as shown by the honest balanced accuracy of 0.5132.
In addition, the query-level warehouse table begins on 2026-04-02, so
query-level features cannot be used for the March 2026 development slice
without creating a temporal mismatch.

Whatever number you just got: interrogate it before you believe it. Which feature carries the
signal? Does it survive a per-client split (train on some clients, test on others)? That
question — *does it generalize across clients?* — is exactly what separates a capstone-grade
result from a lucky split.

## Your turn

1. Re-run section 3 with a **90-day** window and a `HAVING` threshold of your choice.
2. Add one feature you believe in (position volatility? weekend share? query concentration?).
3. Replace the random split with **GroupShuffleSplit on `client_hash_id`** and compare.

## Working locally instead

```python
from huggingface_hub import snapshot_download
path = snapshot_download(repo_id='FlyRank/internship-warehouse', repo_type='dataset',
                         allow_patterns=['dim_*.parquet', 'fact_content_query_90d.parquet',
                                         'fact_content_daily_performance/month=2026-0*/*.parquet'])
```
Then point `REL` at that local path. Download only the month partitions you need — the
`allow_patterns` filter above is the whole trick.

---

**Where this fits:** every lane brief assumes you can produce per-content feature tables like
the one you just built. The lane datasets under the `lanes` HF repo are pre-cut examples of
exactly this pattern — but for the capstone, features you engineered yourself from the full
release beat any pre-cut file.
